# RePaint (part'sız) — kontrol notebook'u

Bu, [`RePaint_part_colab.ipynb`](RePaint_part_colab.ipynb)'nin **kontrol grubu**. Boru hattı,
veri, geometri, metrik, seed'ler **birebir aynı**; tek fark: difüzyon modeli **parçalardan
hiç haberdar değil**.

**Neden gerekli.** "Part-based" bir katkı iddia ediyorsak, katkının olmadığı hâlin ne yaptığını
ölçmek zorundayız. İki olası sonuç, ikisi de bilgi:

* Part'sız RePaint zaten yeterliyse → parça makinesi gereksiz karmaşıklık, at gitsin.
* Part'sız RePaint yetersizse → parça koşullaması **ölçülmüş** bir katkı, hikâye kurulur.

**Yöntemden çıkarılanlar:**

| part notebook'unda | burada |
|---|---|
| denoiser girdisinde parça one-hot | **yok** |
| `part_pool` (parça içi havuzlama) | **yok** — sadece EdgeConv + global havuz |
| parça bazlı dayanak muhasebesi / *anchorless* tespiti | **yok** |
| görünürlüğe uyarlanır resampling bütçesi | **yok** — sabit `jump_n_sample` |
| iki DDPM (part + vanilla) | **tek DDPM** |

**Değerlendirmede kasıtlı olarak KALANLAR** — yoksa iki tablo karşılaştırılamaz:

* `parça-ortalaması` baseline'ı (senin mevcut yöntemin, yenilmesi gereken şey).
* "Tamamen kapalı bölge" senaryosu. Burada GT parça etiketi sadece **hangi bölgenin
  kapatılacağını tanımlamak** için kullanılıyor — yönteme hiçbir bilgi vermiyor.
  Kontrol grubunun test edileceği koşul, deney grubununkiyle aynı olmak zorunda.

Tamamen parçasız çalıştırmak istersen PointNet hücresindeki `PART_BASELINES = False` yeter;
o zaman PointNet hiç eğitilmez ve tablodan `parça-ortalaması` satırı düşer.

---
**Çalıştırma:** Runtime ▸ GPU, sonra Run all. Part notebook'undan **belirgin şekilde hızlı**
(tek DDPM eğitiliyor, değerlendirmede model başına 3 yerine 1 varyant, uyarlanır zıplama yok)
— T4'te uçtan uca **~20 dk** civarı.

---
# A · Ortam kurulumu
> `PoinTr_setup_colab.ipynb` ve `RePaint_part_colab.ipynb` ile **birebir aynı**.

In [ ]:
!nvidia-smi -L
# GPU görünmüyorsa: Runtime > Change runtime type > Hardware accelerator = GPU

### 1) Python bağımlılıkları (numpy<2 sabit; open3d/timm güncel)

In [ ]:
# torch 2.11 numpy 2.x'e karşı derli -> numpy'yi DOWNGRADE ETME (mixed-install -> mtrand ABI hatası).
# Pinli eski open3d==0.9 / timm==0.4.5 py3.12'de derlenmez; güncelleri kurulur.
!pip install -q easydict h5py matplotlib opencv-python pyyaml scipy \
    tensorboardX tqdm transforms3d einops timm open3d gdown
import numpy as np; print("deps OK | numpy", np.__version__, "(2.x olmalı)")

> ⚠️ **numpy'yi 2.x'te bırak** (torch 2.11 onu ister). Eğer `numpy.dtype size changed` /
> `mtrand` hatası alırsan numpy karışmış demektir → şunu çalıştır ve **Runtime ▸ Restart**:
> `!pip install --force-reinstall --no-cache-dir "numpy==2.0.2"`  — sonra baştan çalıştır.

### 2) Fork'u klonla → `/content/PoinTr`

In [ ]:
import os
os.chdir("/content")
if not os.path.isdir("/content/PoinTr"):
    !git clone https://github.com/eylulpelinkilic/Pelin_Efe_PoinTr.git /content/PoinTr
%cd /content/PoinTr
!git rev-parse --short HEAD

### 3) CUDA build ortamı (sabit değil — otomatik tespit + doğru GPU arch)

In [ ]:
import os, glob, torch
# Colab'da aktif toolkit /usr/local/cuda sembolik linkidir; sürümü hardcode ETME
cuda_home = "/usr/local/cuda" if os.path.isdir("/usr/local/cuda") else sorted(glob.glob("/usr/local/cuda*"))[-1]
os.environ["CUDA_HOME"] = cuda_home
os.environ["PATH"] = f"{cuda_home}/bin:" + os.environ["PATH"]
# eklentiler DOĞRU GPU mimarisi için derlensin (T4=7.5, V100=7.0, A100=8.0, L4=8.9) -> "no kernel image" hatasını önler
cap = torch.cuda.get_device_capability(0)
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{cap[0]}.{cap[1]}"
print("torch", torch.__version__, "| torch-cuda", torch.version.cuda,
      "| CUDA_HOME", cuda_home, "| arch", os.environ["TORCH_CUDA_ARCH_LIST"])
!nvcc --version | tail -2

### 4) `pointnet2_ops` (saf-PyTorch shim — derleme yok)
PoinTr'ın `fps`/`three_nn` gibi ops'ları buna bağlı. Çok yeni torch'ta eski CUDA
repo'su derlenmediği için, kullanılan 6 fonksiyonu saf torch'la enjekte ediyoruz.

In [ ]:
# pointnet2_ops'u DERLEMEK yerine saf-PyTorch SHIM olarak enjekte ediyoruz.
# torch 2.11+cu128 gibi çok yeni stack'te eski CUDA repo'su derlenmiyor. PoinTr sadece
# şu 6 fonksiyonu kullanıyor; hepsi saf torch'la doğru (yerelde brute-force'a karşı test edildi).
# FPS saf-torch döngüsü biraz yavaş ama bu ölçekte (~8k nokta) sorun değil.
import sys, types, torch

def furthest_point_sample(xyz, npoint):        # xyz (B,N,3) -> idx (B,npoint) int32
    B, N, _ = xyz.shape; dev = xyz.device
    idx = torch.zeros(B, npoint, dtype=torch.long, device=dev)
    dist = torch.full((B, N), 1e10, device=dev, dtype=xyz.dtype)
    far = torch.zeros(B, dtype=torch.long, device=dev); ar = torch.arange(B, device=dev)
    for i in range(npoint):
        idx[:, i] = far
        dist = torch.minimum(dist, ((xyz - xyz[ar, far].unsqueeze(1)) ** 2).sum(-1))
        far = torch.max(dist, dim=1).indices
    return idx.int()

def gather_operation(features, idx):           # (B,C,N),(B,S) -> (B,C,S)
    B, C, N = features.shape; idx = idx.long()
    return torch.gather(features, 2, idx.unsqueeze(1).expand(B, C, idx.shape[1])).contiguous()

def three_nn(query, ref):                      # (B,N,3),(B,M,3) -> dist(B,N,3) öklid, idx(B,N,3)
    d = torch.cdist(query, ref)
    dist, idx = torch.topk(d, 3, dim=-1, largest=False, sorted=True)
    return dist.contiguous(), idx.int().contiguous()

def three_interpolate(features, idx, weight):  # (B,C,M),(B,N,3),(B,N,3) -> (B,C,N)
    B, C, M = features.shape; N = idx.shape[1]; idx = idx.long()
    g = torch.gather(features, 2, idx.reshape(B,1,N*3).expand(B,C,N*3)).reshape(B,C,N,3)
    return (g * weight.unsqueeze(1)).sum(-1).contiguous()

def grouping_operation(features, idx):         # (B,C,N),(B,S,K) -> (B,C,S,K)  (SnowFlakeNet için)
    B, C, N = features.shape; _, S, K = idx.shape; idx = idx.long()
    return torch.gather(features, 2, idx.reshape(B,1,S*K).expand(B,C,S*K)).reshape(B,C,S,K).contiguous()

def ball_query(radius, nsample, xyz, new_xyz): # (r,k,(B,N,3),(B,S,3)) -> idx(B,S,k)  (SnowFlakeNet için)
    B, N, _ = xyz.shape; S = new_xyz.shape[1]
    d = torch.cdist(new_xyz, xyz)
    idx = torch.arange(N, device=xyz.device).view(1,1,N).expand(B,S,N).contiguous()
    idx[d > radius] = N
    idx = idx.sort(dim=-1).values[:, :, :nsample]
    first = idx[:, :, 0:1].clone(); first[first == N] = 0
    idx = torch.where(idx == N, first.expand(-1,-1,nsample), idx)
    return idx.int()

_u = types.ModuleType("pointnet2_ops.pointnet2_utils")
for _f in [furthest_point_sample, gather_operation, three_nn, three_interpolate,
           grouping_operation, ball_query]:
    setattr(_u, _f.__name__, _f)
_p = types.ModuleType("pointnet2_ops"); _p.pointnet2_utils = _u
sys.modules["pointnet2_ops"] = _p
sys.modules["pointnet2_ops.pointnet2_utils"] = _u
from pointnet2_ops import pointnet2_utils
print("pointnet2_ops shim enjekte edildi:",
      [n for n in dir(pointnet2_utils) if not n.startswith("_")])

### 5) CUDA extension'ları derle (`chamfer` zorunlu; gridding/cubic GRNet için)

In [ ]:
import subprocess
EXTS = ["chamfer_dist", "gridding", "gridding_loss", "cubic_feature_sampling"]  # emd PoinTr için gerekmez
for ext in EXTS:
    print(f"── building {ext} ──")
    # --no-build-isolation: bu setup.py'ler de torch.utils.cpp_extension'a bağlı
    r = subprocess.run("pip install -q --no-build-isolation .", shell=True,
                       cwd=f"/content/PoinTr/extensions/{ext}", capture_output=True, text=True)
    ok = r.returncode == 0
    print("   ", "✅ ok" if ok else "‼ FAILED")
    if not ok:
        print(r.stdout[-600:]); print(r.stderr[-1800:])

### 6) Her şey import oluyor mu? (GPU smoke test)

In [ ]:
import os, sys
sys.path.insert(0, "/content/PoinTr"); os.chdir("/content/PoinTr")
import torch, numpy as np
print("numpy", np.__version__, "| torch", torch.__version__)
import chamfer, gridding, gridding_distance, cubic_feature_sampling
from pointnet2_ops import pointnet2_utils
from extensions.chamfer_dist import ChamferDistanceL1
from models.PoinTr import PoinTr, Fold, fps
from models.dgcnn_group import DGCNN_Grouper
from models.Transformer import PCTransformer
# gerçekten GPU'da çalışıyor mu: fps + chamfer
x = torch.rand(1, 1024, 3, device="cuda")
idx = pointnet2_utils.furthest_point_sample(x, 128)
d = ChamferDistanceL1()(x, torch.rand(1, 512, 3, device="cuda"))
print("pointnet2 fps:", tuple(idx.shape), "| chamfer:", float(d))
print("✅ PoinTr environment READY")

### 7) Pretrained checkpoint (ShapeNet55)

In [ ]:
import os, subprocess
CKPT = "/content/PoinTr/ckpts/PoinTr_ShapeNet55.pth"
os.makedirs(os.path.dirname(CKPT), exist_ok=True)
if not os.path.exists(CKPT) or os.path.getsize(CKPT) < 50e6:
    subprocess.run(f"gdown 1WzERLlbSwzGOBybzkjBrApwyVMTG00CJ -O {CKPT}", shell=True, check=True)
print("checkpoint MB:", round(os.path.getsize(CKPT)/1e6, 1), " (>400 olmalı)")

---
# B · Veri — part notebook'uyla **aynı** (otomatik iner)

Aynı kaynak, aynı `N_MODELS`, aynı `CROP`, aynı train/test ayrımı, aynı seed'ler.
İki notebook'un tablosunun karşılaştırılabilir olması buna bağlı — burayı değiştirirsen
diğerinde de aynısını değiştir.

In [ ]:
# --- yardımcılar (PoinTr_setup_colab.ipynb ile aynı) ---
import numpy as np, torch, os, glob, zipfile

def _crop_order(xyz, seed):
    """Rastgele bir bakış yönüne göre noktaları sırala (yakın olanlar kırpılacak).
       Yön SADECE seed'e bağlı -> aynı nesnenin farklı çözünürlükleri AYNI bölgeden kırpılır."""
    c = xyz.mean(0); n = (xyz - c) / (np.linalg.norm(xyz - c, axis=1).max() + 1e-9)
    rng = np.random.default_rng(seed); v = rng.standard_normal(3); v /= np.linalg.norm(v)
    return np.argsort(np.linalg.norm(n - v[None], axis=1))

def separate_colored(gt, crop=0.5, seed=0):     # -> partial(6D), mask(True=eksik)
    order = _crop_order(gt[:, :3], seed); N = len(gt); nc = int(round(N * crop))
    mask = np.zeros(N, bool); mask[order[:nc]] = True
    return gt[order[nc:]], mask

def crop_xyz(xyz, crop=0.5, seed=0):
    """separate_colored ile AYNI bölgeyi, sadece xyz olarak, NATIVE çözünürlükte kırp.
       PoinTr'a mümkün olan en yoğun girdiyi verebilmek için."""
    order = _crop_order(xyz, seed); nc = int(round(len(xyz) * crop))
    return np.ascontiguousarray(xyz[order[nc:]]).astype(np.float32)

def srgb_to_lab(rgb):
    rgb = np.clip(rgb, 0, 1); lin = np.where(rgb > 0.04045, ((rgb + 0.055) / 1.055) ** 2.4, rgb / 12.92)
    M = np.array([[0.4124,0.3576,0.1805],[0.2126,0.7152,0.0722],[0.0193,0.1192,0.9505]])
    xyz = (lin @ M.T) / np.array([0.95047, 1.0, 1.08883]); d = 6 / 29
    f = np.where(xyz > d ** 3, np.cbrt(xyz), xyz / (3 * d ** 2) + 4 / 29)
    return np.stack([116*f[:,1]-16, 500*(f[:,0]-f[:,1]), 200*(f[:,1]-f[:,2])], 1)
def deltaE(a, b): return np.linalg.norm(srgb_to_lab(a) - srgb_to_lab(b), axis=1)
print("yardımcılar hazır")

In [ ]:
# ================= VERİ AYARLARI (hepsi otomatik — dokunmana gerek yok) =================
DATA_SOURCE = "auto"        # "auto" (indirir) | "labeled_s3" (Aşama 1) | "partanno" (yerel)
CATEGORY    = "Airplane"    # "Airplane" | "Chair" | "Car" | "Table" | ... (SEG_CLASSES'taki her şey)
N_PTS, CROP, N_MODELS, CNOISE = 2048, 0.5, 150, 0.03
TEST_FRAC   = 0.25          # DDPM + PointNet SADECE train bölümünde eğitilir
HF_REPO     = "larryshaw0079/ShapeNetPart"   # açık mirror, TOKEN GEREKMEZ
HF_FILES    = ["train0.h5", "test1.h5"]      # ~118 MB toplam
# labeled_s3 nerede aranacak. SIRAYLA denenir, ilk bulunan kullanılır.
# /content runtime kopunca SİLİNİR -> zip'i bir kez Drive'a koy, bir daha uğraşma.
LABELED_CANDIDATES = [
    "/content/labeled_s3_02691156.zip",
    "/content/drive/MyDrive/labeled_s3_02691156.zip",
    "/content/drive/MyDrive/pcc_ckpt/labeled_s3_02691156.zip",
    "/content/labeled_s3", "/content/drive/MyDrive/labeled_s3",
]
LABELED_EXTRACT = "/content/_labeled"
TRY_DRIVE_FOR_DATA = True     # hiçbiri yoksa Drive'ı bağlayıp tekrar bak

def find_labeled(mount=TRY_DRIVE_FOR_DATA):
    """İlk var olan labeled_s3 kaynağını döndür; yoksa Drive'ı bağlayıp bir daha bak."""
    for p in LABELED_CANDIDATES:
        if os.path.exists(p): return p
    if mount and not os.path.isdir("/content/drive"):
        try:
            from google.colab import drive; drive.mount("/content/drive")
        except Exception:
            return None
        for p in LABELED_CANDIDATES:
            if os.path.exists(p): return p
    return None
PART_SRC    = "/content/PartAnnotation/02691156"      # sadece DATA_SOURCE="partanno" için
EXTRACT     = "/content/_spart"
# =======================================================================================

# ShapeNet-Part standardı: kategoriler alfabetik (hdf5 'label' = bu listedeki indeks),
# her kategorinin parçaları global 0..49 aralığında ardışık.
SEG_CLASSES = {"Airplane": [0,1,2,3], "Bag": [4,5], "Cap": [6,7], "Car": [8,9,10,11],
               "Chair": [12,13,14,15], "Earphone": [16,17,18], "Guitar": [19,20,21],
               "Knife": [22,23], "Lamp": [24,25,26,27], "Laptop": [28,29],
               "Motorbike": [30,31,32,33,34,35], "Mug": [36,37], "Pistol": [38,39,40],
               "Rocket": [41,42,43], "Skateboard": [44,45,46], "Table": [47,48,49]}
PART_NAMES_BY_CAT = {"Airplane": ["body", "wing", "tail", "engine"],   # sadece görüntüleme için
                     "Chair": ["back", "seat", "leg", "arm"]}
PALETTE = np.array([[.85,.20,.20],[.20,.65,.25],[.20,.35,.80],[.90,.75,.20],
                    [.60,.30,.75],[.25,.70,.70],[.90,.55,.20],[.5,.5,.5]])

def _finalize(xyz, part, P, j, rng):
    """-> (xyz@N_PTS, rgb, part, xyz@NATIVE). Önce normalize, SONRA örnekle: ikisi de
       aynı dönüşümde olsun ki dense kırpma ile seyrek kırpma aynı bölgeyi versin."""
    xyz = xyz - xyz.mean(0); xyz = xyz / (np.linalg.norm(xyz, axis=1).max() + 1e-9)
    dense = np.ascontiguousarray(xyz).astype(np.float32)          # native çözünürlük (PoinTr için)
    if len(xyz) != N_PTS:
        s = np.random.default_rng(j).choice(len(xyz), N_PTS, replace=len(xyz) < N_PTS)
        xyz, part = xyz[s], part[s]
    rgb = np.clip(PALETTE[:P][part] + rng.normal(0, CNOISE, (N_PTS, 3)), 0, 1).astype(np.float32)
    return xyz.astype(np.float32), rgb, part, dense

def _load_auto():
    """ShapeNet-Part HDF5 benchmark'ını HF'ten indir (tokensız) ve kategoriyi süz."""
    import h5py, urllib.request
    assert CATEGORY in SEG_CLASSES, f"bilinmeyen kategori: {CATEGORY}"
    cat_id, gids = list(SEG_CLASSES).index(CATEGORY), SEG_CLASSES[CATEGORY]
    P = len(gids)
    cache = "/content/_snpart"; os.makedirs(cache, exist_ok=True)
    X, S = [], []
    for fn in HF_FILES:
        p = os.path.join(cache, fn)
        if not os.path.exists(p):
            print(f"  indiriliyor {fn} ...", end=" ", flush=True)
            urllib.request.urlretrieve(f"https://huggingface.co/datasets/{HF_REPO}/resolve/main/{fn}", p)
            print("ok")
        with h5py.File(p, "r") as f:
            m = f["label"][:].ravel() == cat_id
            if m.any(): X.append(f["data"][:][m]); S.append(f["seg"][:][m])
        if sum(len(a) for a in X) >= N_MODELS: break
    assert X, f"{CATEGORY} bulunamadı ({HF_FILES})"
    xyz, seg = np.concatenate(X)[:N_MODELS], np.concatenate(S)[:N_MODELS] - gids[0]
    names = (PART_NAMES_BY_CAT.get(CATEGORY) or [f"part{i}" for i in range(P)])[:P]
    rng = np.random.default_rng(0)
    return [_finalize(xyz[j].astype(np.float32), seg[j].astype(np.int64), P, j, rng)
            for j in range(len(xyz))], names, P

def _load_partanno():
    def _resolve(src):
        if os.path.isdir(src): return src
        if zipfile.is_zipfile(src):
            if not os.path.isdir(EXTRACT):
                with zipfile.ZipFile(src) as z: z.extractall(EXTRACT)
            return EXTRACT
        raise FileNotFoundError(f"'{src}' ne klasör ne geçerli zip. /content: {os.listdir('/content')[:20]}")
    ROOT = _resolve(PART_SRC)
    hit = glob.glob(f"{ROOT}/**/points/*.pts", recursive=True); assert hit, f"points/*.pts yok {ROOT}"
    PART_DIR = os.path.dirname(os.path.dirname(hit[0]))
    subs = sorted(d for d in glob.glob(os.path.join(PART_DIR, "points_label", "*")) if os.path.isdir(d))
    names = [os.path.basename(d) for d in subs]; P = len(subs)
    assert P >= 2, f"parça alt-klasörü yok: {PART_DIR}/points_label/"
    pts = sorted(glob.glob(os.path.join(PART_DIR, "points", "*.pts")))
    rng = np.random.default_rng(0)
    out = []
    for j in range(min(N_MODELS, len(pts))):
        p = pts[j]; mid = os.path.splitext(os.path.basename(p))[0]
        xyz = np.loadtxt(p, dtype=np.float32)[:, :3]; lab = np.zeros(len(xyz), np.int64)
        for pi, pd in enumerate(subs):
            sf = os.path.join(pd, mid + ".seg")
            if os.path.exists(sf):
                v = np.loadtxt(sf)
                if len(v) == len(xyz): lab[v != 0] = pi
        out.append(_finalize(xyz, lab, P, j, rng))
    return out, names, P

def _load_labeled_s3():
    """GERÇEK dokulu GT: scripts/build_gt.py + build_labeled_gt.py çıktısı.

    Kaynak zip DA olabilir klasör de; LABELED_CANDIDATES sırayla /content ve Drive'a bakar.
    ÖNERİ: zip'i Drive'a (MyDrive kökü) koy — /content runtime kopunca siliniyor.
    Renkler ShapeNet'in dokulu mesh'inden örneklenmiş GERÇEK renkler; parça etiketleri
    standart ShapeNet-Part sırasında (body, wing, tail, engine) — 'auto' kaynağıyla aynı
    konvansiyon, yani iki kaynağın parça indeksleri birbirini tutar.
    """
    src = find_labeled()
    assert src, ("labeled_s3 bulunamadı. Arananlar:\n  " + "\n  ".join(LABELED_CANDIDATES) +
                 "\n  scripts/build_gt.py -> build_labeled_gt.py ile üret, zip'le, "
                 "Drive'a (MyDrive kökü) koy.")
    if zipfile.is_zipfile(src):
        if not os.path.isdir(LABELED_EXTRACT):
            with zipfile.ZipFile(src) as z: z.extractall(LABELED_EXTRACT)
        src = LABELED_EXTRACT
    files = sorted(glob.glob(os.path.join(src, "**", "*.npz"), recursive=True))
    assert files, f"npz bulunamadı: {src}"
    P = len(SEG_CLASSES.get(CATEGORY, [0])) or 1
    names = (PART_NAMES_BY_CAT.get(CATEGORY) or [f"part{i}" for i in range(P)])[:P]
    out = []
    for j, f in enumerate(files[:N_MODELS]):
        z = np.load(f)
        xyz, rgb, part = z["xyz"], z["rgb"], z["part"].astype(np.int64)
        xyz = xyz - xyz.mean(0); xyz = xyz / (np.linalg.norm(xyz, axis=1).max() + 1e-9)
        dense = np.ascontiguousarray(xyz).astype(np.float32)   # 8192 nokta -> PoinTr'a yoğun girdi
        if len(xyz) != N_PTS:
            s = np.random.default_rng(j).choice(len(xyz), N_PTS, replace=len(xyz) < N_PTS)
            xyz, rgb, part = xyz[s], rgb[s], part[s]
        out.append((xyz.astype(np.float32), np.clip(rgb, 0, 1).astype(np.float32),
                    np.clip(part, 0, P - 1), dense))
    print(f"  labeled_s3: {len(out)} model (GERÇEK doku), parçalar {names}")
    return out, names, P

# labeled_s3 zip'i yüklüyse Aşama 1 istiyorsun demektir -> otomatik geç.
# ("DATA_SOURCE'u değiştirmeyi unutup 30 dk sentetik çalıştırmak" bir kez oldu, bir daha olmasın.)
_found = find_labeled() if DATA_SOURCE == "auto" else None
if _found:
    DATA_SOURCE = "labeled_s3"
    print(f"ℹ️  labeled_s3 bulundu -> {_found}")
    print("    DATA_SOURCE otomatik 'labeled_s3' yapıldı (bilerek sentetik istiyorsan zip'i kaldır).")
elif DATA_SOURCE == "auto":
    print("ℹ️  labeled_s3 bulunamadı, SENTETİK renkle devam. Arananlar:")
    for _p in LABELED_CANDIDATES: print("      ", _p)

_LOADERS = {"auto": _load_auto, "partanno": _load_partanno, "labeled_s3": _load_labeled_s3}
assert DATA_SOURCE in _LOADERS, f"DATA_SOURCE {DATA_SOURCE!r} olmaz, seçenekler: {list(_LOADERS)}"
raw, PART_NAMES, NUM_PARTS = _LOADERS[DATA_SOURCE]()

DATA = []
for j, (xyz, rgb, part, dense) in enumerate(raw):
    gt = np.concatenate([xyz, rgb], 1).astype(np.float32)
    partial, miss = separate_colored(gt, CROP, seed=j)
    # AYNI bakış yönüyle native çözünürlükte kırp -> PoinTr'a verilecek yoğun partial.
    # (native == N_PTS ise kazanç yok, None kalır ve seyrek partial kullanılır.)
    pdense = crop_xyz(dense, CROP, seed=j) if len(dense) > len(gt) else None
    DATA.append(dict(gt=gt, partial=partial, miss=miss, gt_part=part, partial_dense=pdense))
_pd = DATA[0]["partial_dense"]
print(f"PoinTr girdisi: partial {len(DATA[0]['partial'])} nokta"
      + (f"  |  YOĞUN partial {len(_pd)} nokta (native {len(raw[0][3])})" if _pd is not None
         else f"  |  yoğun sürüm YOK (kaynak zaten {len(raw[0][3])} nokta)"))

# train/test ayrımı — DDPM ve PointNet SADECE TRAIN'de eğitilir, tablo TEST'te ölçülür
n_test = max(1, int(len(DATA) * TEST_FRAC))
TEST_IDX  = list(range(len(DATA) - n_test, len(DATA)))
TRAIN_IDX = list(range(len(DATA) - n_test))
print(f"DATA: {len(DATA)} model | {N_PTS} nokta | {NUM_PARTS} parça {PART_NAMES}")
print(f"kaynak={DATA_SOURCE} | train {len(TRAIN_IDX)} / test {len(TEST_IDX)} (tablo test'te)")

# --- HANGİ VERİYLE ÇALIŞTIĞINI ÖLÇEREK SÖYLE ---------------------------------
# Yanlış kaynakla 30 dk çalıştırıp sonunda fark etmek çok kolay. Parça içi ΔE bunu
# tek sayıda ele veriyor: sentetik ~5, gerçek doku ~20.
_ce = []
for _d in DATA[:min(30, len(DATA))]:
    _g, _p = _d["gt"], _d["gt_part"]
    for _k in range(NUM_PARTS):
        _m = _p == _k
        if _m.sum() > 10:
            _ce.append(deltaE(_g[_m, 3:6], np.tile(_g[_m, 3:6].mean(0), (_m.sum(), 1))).mean())
CEIL = float(np.mean(_ce))
print("=" * 70)
if DATA_SOURCE == "labeled_s3":
    print("  RENK: ✅ GERÇEK DOKU (ShapeNet mesh'inden örneklenmiş)")
else:
    print("  RENK: ⚠️  SENTETİK — PALETTE[parça] + gürültü.  BU AŞAMA 0.")
    print("        Gerçek doku için: labeled_s3 zip'ini yükle + DATA_SOURCE='labeled_s3'")
print(f"  parça içi ΔE = {CEIL:.1f}   (= parça-ortalamasının yapısal TAVANI)")
if CEIL < 10:
    print("  -> Düz renk. Parça-ortalaması burada zaten ~optimal; RePaint onu GEÇEMEZ.")
    print("     Beklenen sonuç: tavana yaklaşmak + NN'i ezmek + kapalı parçayı kurtarmak.")
else:
    print("  -> Dokulu. Parça-ortalaması bu değerin ALTINA inemez; RePaint inebilir.")
    print("     Asıl karşılaştırma bu: parça-ortalaması tavana yapışacak mı, RePaint altına inecek mi.")
print("=" * 70)

---
# C · Dondurulmuş PoinTr + (opsiyonel) part-seg

Geometri tarafı aynen aynı. PointNet part-seg burada **yönteme değil, sadece
`parça-ortalaması` baseline'ına** hizmet ediyor.

In [ ]:
# --- STEP 1: dondurulmuş orijinal PoinTr ---
import torch
from easydict import EasyDict
from scipy.spatial import cKDTree
from models.PoinTr import PoinTr, fps

DEV = "cuda"
cfg = EasyDict(trans_dim=384, knn_layer=1, num_pred=6144, num_query=96)
geo = PoinTr(cfg)
sd = torch.load(CKPT, map_location="cpu")
base = sd.get("base_model", sd.get("model", sd))
base = {k.replace("module.", ""): v for k, v in base.items()}
mi, ui = geo.load_state_dict(base, strict=False)
assert len(mi) == 0, f"orijinal PoinTr bekleniyordu, missing={mi[:4]} (0 olmalı)"
for p in geo.parameters(): p.requires_grad_(False)
geo.eval().to(DEV)

# ShapeNet-Part -> PoinTr(ShapeNet-55) frame hizası. Aşağıdaki hücre 48 işaretli
# permütasyonu Chamfer ile tarayıp bunları OTOMATİK ayarlıyor — elle dokunma.
AXIS_PERM, AXIS_SIGN = (2, 1, 0), (1, 1, 1)

@torch.no_grad()
def complete_geometry(partial, perm=None, sign=None):
    """RENKSİZ partial (xyz) -> PoinTr tamamlaması, doğru frame'e çevirip geri çevirerek."""
    perm = AXIS_PERM if perm is None else tuple(perm)
    sign = AXIS_SIGN if sign is None else tuple(sign)
    s = np.asarray(sign, np.float32)
    x = np.ascontiguousarray(partial[:, :3][:, list(perm)] * s)      # x'[:,i] = x[:,perm[i]]*s[i]
    p = torch.from_numpy(x).float().unsqueeze(0).to(DEV)
    fine = geo(p)[1][0, :geo.num_pred].cpu().numpy()
    inv = np.argsort(perm)                                          # ters çevir: x[:,j] = x'[:,inv[j]]*s[inv[j]]
    return np.ascontiguousarray(fine[:, inv] * s[inv])

def chamfer_l1(a, b):
    """Simetrik ortalama en-yakın-komşu mesafesi (düşük = iyi)."""
    d1, _ = cKDTree(b).query(a, k=1)
    d2, _ = cKDTree(a).query(b, k=1)
    return float(d1.mean() + d2.mean())

# PoinTr ShapeNet-55, 8192 noktalı bulutlardan kırpılmış 2048-6144 noktalı partial'larla
# eğitildi. Bizim seyrek partial'ımız (N_PTS=2048, CROP=0.5 -> 1024 nokta) o dağılımın
# ALTINDA kalıyor ve DGCNN grouper'ın kNN komşulukları ~2x geniş düşüyor. Bu yüzden
# PoinTr'a verilen girdiyi renk hattından AYIRIYORUZ: aşağıdaki hücre en iyisini ölçüyor.
POINTR_IN = 2048          # PoinTr'a verilecek nokta sayısı (yoğun partial varsa FPS ile)
USE_DENSE_FOR_POINTR = True

def _fps_np(xyz, n):
    """PoinTr'ın kendi fps'i (pointnet2_ops shim'i üzerinden)."""
    if len(xyz) <= n:
        return np.ascontiguousarray(xyz).astype(np.float32)
    t = torch.from_numpy(np.ascontiguousarray(xyz[:, :3])).float().unsqueeze(0).to(DEV)
    return fps(t, n)[0].cpu().numpy().astype(np.float32)

def pointr_input(d):
    """Bu model için PoinTr'a verilecek xyz — mümkünse yoğun partial, POINTR_IN'e indirilmiş."""
    src = d.get("partial_dense") if USE_DENSE_FOR_POINTR else None
    src = d["partial"][:, :3] if src is None else src
    return _fps_np(src, POINTR_IN) if POINTR_IN and len(src) > POINTR_IN else src

def complete_of(d, perm=None, sign=None):
    """Bu modelin PoinTr tamamlaması. Boru hattının HER yeri bunu kullanmalı."""
    return complete_geometry(pointr_input(d), perm, sign)

def nn_color(partial, comp):                   # BASELINE 1
    _, i = cKDTree(partial[:, :3]).query(comp[:, :3], k=1)
    return partial[i, 3:6]
print(f"dondurulmuş PoinTr hazır | num_pred={geo.num_pred}")

### Oryantasyon — **48 permütasyonu tarayıp otomatik seçer**

PoinTr **ShapeNet-55** frame'inde eğitildi; ShapeNet-Part farklı eksen konvansiyonunda.
Yanlış frame verilirse PoinTr şekli tanımaz ve **dağınık bir blob** üretir.

Eskiden burada 3 permütasyon çizilip göze bırakılıyordu — yetersizdi: doğru dönüşüm bir işaret
çevirmesi de içerebilir, yani **6 permütasyon × 8 işaret = 48 aday** var ve 3'ü bakmak yanıltıcı.
GT elimizde olduğu için tahmin etmeye gerek yok: hepsini deneyip **Chamfer** ile ölçüyoruz.

Hücre ayrıca **PoinTr'ın işe yarayıp yaramadığını** söylüyor. Referans olarak `partial → GT`
Chamfer'ını da basıyor: en iyi tamamlama bundan iyi değilse sorun oryantasyonda değildir
(o zaman girdi yoğunluğu / nokta sayısı şüphelisi — hücre onu da uyarıyor).

In [ ]:
# --- oryantasyon: 48 işaretli permütasyonu Chamfer ile tara, en iyisini SEÇ ---
import itertools
import plotly.graph_objects as go
from plotly.subplots import make_subplots

N_PROBE = 3                      # kaç model üzerinden ortalama (48 x N_PROBE PoinTr geçişi)
_probe = [DATA[i] for i in TRAIN_IDX[:N_PROBE]]
_cands = [(p, s) for p in itertools.permutations(range(3))
                 for s in itertools.product((1, -1), repeat=3)]

# referans: hiç tamamlama yapmadan, ham partial'ın GT'ye Chamfer'ı
_ref_partial = float(np.mean([chamfer_l1(d["partial"][:, :3], d["gt"][:, :3]) for d in _probe]))

# ---- 1) FRAME: 48 işaretli permütasyon ----
_scores = []
for _perm, _sign in _cands:
    _c = [chamfer_l1(complete_of(d, _perm, _sign), d["gt"][:, :3]) for d in _probe]
    _scores.append((float(np.mean(_c)), _perm, _sign))
_scores.sort()
_best, AXIS_PERM, AXIS_SIGN = _scores[0]
print(f"[1] FRAME taraması (girdi: {'yoğun' if USE_DENSE_FOR_POINTR else 'seyrek'}, "
      f"{len(pointr_input(_probe[0]))} nokta)")
print(f"    {'sıra':>4}  {'perm':<10} {'işaret':<12} Chamfer")
for _i, (_v, _p, _s) in enumerate(_scores[:5], 1):
    print(f"    {_i:>4}  {str(list(_p)):<10} {str(list(_s)):<12} {_v:.4f}")
print(f"    {'':>4}  {'(en kötü)':<10} {str(list(_scores[-1][2])):<12} {_scores[-1][0]:.4f}")
print(f"    -> AXIS_PERM={list(AXIS_PERM)}  AXIS_SIGN={list(AXIS_SIGN)}")

# ---- 2) GİRDİ YOĞUNLUĞU: PoinTr'a kaç nokta vermeli? ----
_nd = len(_probe[0]["partial_dense"]) if _probe[0].get("partial_dense") is not None else 0
_vars = [("seyrek", len(_probe[0]["partial"]), False)]
_vars += [("yoğun", _n, True) for _n in sorted({1024, 2048, 4096, _nd}) if 0 < _n <= _nd]
_dres = []
for _kind, _n, _dense in _vars:
    USE_DENSE_FOR_POINTR, POINTR_IN = _dense, _n
    _c = [chamfer_l1(complete_of(d, AXIS_PERM, AXIS_SIGN), d["gt"][:, :3]) for d in _probe]
    _dres.append((float(np.mean(_c)), _kind, _n, _dense))
_dres.sort()
print(f"\n[2] GİRDİ YOĞUNLUĞU taraması")
for _v, _k, _n, _ in _dres:
    print(f"    {_k:<8} {_n:>5} nokta   Chamfer {_v:.4f}")
_best, _bk, POINTR_IN, USE_DENSE_FOR_POINTR = _dres[0]
print(f"    -> POINTR_IN={POINTR_IN}  ({_bk} partial)")
if _nd == 0:
    print("    NOT: kaynak zaten native çözünürlükte, yoğun sürüm üretilemedi.")
    print("         Yoğunluk şüpheliyse DATA_SOURCE='labeled_s3' (8192 nokta) kullan.")

# ---- 3) PoinTr işe yarıyor mu? ----
print(f"\n[3] referans — partial→GT Chamfer (hiç tamamlama yok): {_ref_partial:.4f}")
print(f"    en iyi tamamlama:                                  {_best:.4f}")
if _best > _ref_partial:
    print("    ⛔ Tamamlama ham partial'dan KÖTÜ -> PoinTr bu veride şekli tanımıyor.")
    print("       Ne frame ne yoğunluk çözdü; aşağıdaki figürde çıktı blob görünecek.")
elif _best > 0.6 * _ref_partial:
    print("    ⚠️  Tamamlama partial'dan sadece biraz iyi — şekil hâlâ dağınık olabilir.")
else:
    print("    ✅ Tamamlama partial'dan belirgin iyi — PoinTr şekli tanıyor.")

# görsel: en iyi / en kötü frame / girdi / GT
_d = _probe[0]
_panels = [(f"EN İYİ {list(AXIS_PERM)}{list(AXIS_SIGN)} @{POINTR_IN}",
            complete_of(_d, AXIS_PERM, AXIS_SIGN), "#12a5b8"),
           ("en kötü frame", complete_of(_d, _scores[-1][1], _scores[-1][2]), "#c0554d"),
           (f"PoinTr girdisi ({len(pointr_input(_d))} nk)", pointr_input(_d), "#8a8f96"),
           ("GT (referans)", _d["gt"][:, :3], "#8a8f96")]
_fig = make_subplots(rows=1, cols=4, specs=[[{"type": "scene"}] * 4],
                     subplot_titles=[t for t, _, _ in _panels])
for _c, (_, _xyz, _col) in enumerate(_panels, 1):
    _fig.add_trace(go.Scatter3d(x=_xyz[:,0], y=_xyz[:,1], z=_xyz[:,2], mode="markers",
                                marker=dict(size=1.4, color=_col)), 1, _c)
for _s in _fig.layout:
    if _s.startswith("scene"): _fig.layout[_s].aspectmode = "data"
_fig.update_layout(height=380, showlegend=False, margin=dict(l=0, r=0, t=30, b=0)); _fig.show()

In [ ]:
# --- part-seg: SADECE baseline için (yöntem bunu kullanmıyor) ---
PART_BASELINES = True     # False -> PointNet hiç eğitilmez, tablodan parça-ortalaması düşer

import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

def segment_oracle(d, xyz):
    """En yakın GT noktasının parçası. PointNet GEREKTİRMEZ — sadece KD-tree.
       Burada yalnızca 'tamamen kapalı bölge' senaryosunu TANIMLAMAK için kullanılıyor."""
    x = np.asarray(xyz, np.float32)[:, :3]
    c = x.mean(0); x = (x - c) / (np.linalg.norm(x - c, axis=1).max() + 1e-9)
    g = d["gt"][:, :3]; gc = g.mean(0); g = (g - gc) / (np.linalg.norm(g - gc, axis=1).max() + 1e-9)
    _, j = cKDTree(g).query(x, k=1)
    return d["gt_part"][j]

if PART_BASELINES:
    class _TNet(nn.Module):
        def __init__(s, k):
            super().__init__(); s.k = k
            s.mlp = nn.Sequential(nn.Conv1d(k,64,1),nn.BatchNorm1d(64),nn.ReLU(),
                nn.Conv1d(64,128,1),nn.BatchNorm1d(128),nn.ReLU(),
                nn.Conv1d(128,1024,1),nn.BatchNorm1d(1024),nn.ReLU())
            s.fc = nn.Sequential(nn.Linear(1024,512),nn.ReLU(),nn.Linear(512,256),nn.ReLU(),
                                 nn.Linear(256,k*k))
        def forward(s, x):
            B=x.size(0); f=s.mlp(x).max(-1)[0]
            return s.fc(f).view(B,s.k,s.k) + torch.eye(s.k,device=x.device).unsqueeze(0)

    class PointNetPartSeg(nn.Module):
        def __init__(s, P):
            super().__init__(); s.itn=_TNet(3)
            s.mlp1=nn.Sequential(nn.Conv1d(3,64,1),nn.BatchNorm1d(64),nn.ReLU(),
                                 nn.Conv1d(64,128,1),nn.BatchNorm1d(128),nn.ReLU())
            s.fstn=_TNet(128)
            s.mlp2=nn.Sequential(nn.Conv1d(128,128,1),nn.BatchNorm1d(128),nn.ReLU(),
                                 nn.Conv1d(128,1024,1),nn.BatchNorm1d(1024),nn.ReLU())
            s.seg=nn.Sequential(nn.Conv1d(1152,512,1),nn.BatchNorm1d(512),nn.ReLU(),
                nn.Conv1d(512,256,1),nn.BatchNorm1d(256),nn.ReLU(),nn.Conv1d(256,P,1))
        def forward(s, x):
            x=x.transpose(1,2); x=torch.bmm(s.itn(x),x)
            f=s.mlp1(x); f=torch.bmm(s.fstn(f),f); pf=f
            g=s.mlp2(f).max(-1,keepdim=True)[0].expand(-1,-1,f.size(-1))
            return s.seg(torch.cat([pf,g],1)).transpose(1,2)

    def train_partseg(model, loader, epochs=60, lr=1e-3, device=DEV):
        model.to(device).train(); opt=torch.optim.Adam(model.parameters(),lr)
        lf=nn.CrossEntropyLoss()
        for ep in range(epochs):
            cor=seen=0; tot=0.0
            for xyz,lab in loader:
                xyz,lab=xyz.to(device),lab.to(device); opt.zero_grad()
                lo=model(xyz); loss=lf(lo.reshape(-1,lo.size(-1)),lab.reshape(-1))
                loss.backward(); opt.step()
                tot+=loss.item()*xyz.size(0); cor+=(lo.argmax(-1)==lab).sum().item(); seen+=lab.numel()
            if ep%10==0 or ep==epochs-1:
                print(f"  ep{ep:3d} loss {tot/len(loader.dataset):.4f} acc {cor/seen*100:.1f}%")
        return model

    @torch.no_grad()
    def segment(model, xyz, device=DEV):
        model.eval(); x=np.asarray(xyz,np.float32)[:,:3]
        c=x.mean(0); x=(x-c)/(np.linalg.norm(x-c,axis=1).max()+1e-9)
        return model(torch.from_numpy(x).float().unsqueeze(0).to(device))[0].argmax(-1).cpu().numpy()

    class _GTPartDS(Dataset):
        def __init__(s, D): s.D = D
        def __len__(s): return len(s.D)
        def __getitem__(s, i):
            d = s.D[i]
            return torch.from_numpy(d["gt"][:, :3]).float(), torch.from_numpy(d["gt_part"]).long()

    seg_model = PointNetPartSeg(NUM_PARTS).to(DEV)
    train_partseg(seg_model, DataLoader(_GTPartDS([DATA[i] for i in TRAIN_IDX]),
                                        batch_size=16, shuffle=True), epochs=60)
    _acc = [(segment(seg_model, DATA[i]["gt"][:, :3]) == DATA[i]["gt_part"]).mean() for i in TEST_IDX]
    print(f"\nPointNet part-seg TEST doğruluğu: {np.mean(_acc)*100:.1f}%  (sadece baseline için)")

    def part_color_pointnet(partial, comp, model):     # BASELINE: parça-ortalaması
        vl = segment(model, partial[:, :3]); cl = segment(model, comp[:, :3])
        vr = partial[:, 3:6]
        mean = np.tile(vr.mean(0), (NUM_PARTS, 1))
        for k in range(NUM_PARTS):
            m = vl == k
            if m.any(): mean[k] = vr[m].mean(0)
        return mean[cl]
else:
    seg_model, _acc = None, [float("nan")]
    print("PART_BASELINES=False -> PointNet atlandı, tablo tamamen parçasız.")

---
# D · RePaint — part'sız

`D1` (DDPM şeması + zıplama şeması) part notebook'uyla **birebir aynı** — parçayla ilgisi yok.
`D2`–`D4` ise parça mantığından arındırıldı.

In [ ]:
# ---------------- D1 · DDPM şeması + RePaint zıplama şeması ----------------
import math

def cosine_betas(T, s=0.008):
    """Nichol & Dhariwal cosine schedule — düşük boyutlu sinyalde linear'dan iyi."""
    t = torch.linspace(0, T, T + 1) / T
    f = torch.cos((t + s) / (1 + s) * math.pi / 2) ** 2
    ab = f / f[0]
    return (1 - ab[1:] / ab[:-1]).clamp(1e-8, 0.999)

class Diffusion:
    """Düz DDPM (eps-tahmini), [-1,1] aralığındaki (N,3) renk alanı üzerinde."""
    def __init__(self, T=200, device=DEV):
        self.T, self.device = T, device
        b = cosine_betas(T).to(device); a = 1.0 - b
        abar = torch.cumprod(a, 0)
        abar_prev = torch.cat([torch.ones(1, device=device), abar[:-1]])
        self.betas, self.alphas, self.abar, self.abar_prev = b, a, abar, abar_prev
        self.sqrt_abar, self.sqrt_1mabar = abar.sqrt(), (1 - abar).sqrt()
        self.post_var = b * (1 - abar_prev) / (1 - abar)            # q(x_{t-1}|x_t,x_0)
        self.post_c0  = b * abar_prev.sqrt() / (1 - abar)
        self.post_ct  = (1 - abar_prev) * a.sqrt() / (1 - abar)

    def q_sample(self, x0, t, noise=None):
        noise = torch.randn_like(x0) if noise is None else noise
        sa = self.sqrt_abar[t].view(-1, *([1] * (x0.dim() - 1)))
        sb = self.sqrt_1mabar[t].view(-1, *([1] * (x0.dim() - 1)))
        return sa * x0 + sb * noise

    def p_sample(self, eps, x_t, t, generator=None):
        x0 = ((x_t - self.sqrt_1mabar[t] * eps) / self.sqrt_abar[t]).clamp(-1, 1)
        mean = self.post_c0[t] * x0 + self.post_ct[t] * x_t
        if t == 0: return mean, x0
        z = torch.randn(x_t.shape, device=x_t.device, dtype=x_t.dtype, generator=generator)
        return mean + self.post_var[t].sqrt() * z, x0

    def forward_jump(self, x, t, generator=None):        # RePaint time-travel: x_t -> x_{t+1}
        z = torch.randn(x.shape, device=x.device, dtype=x.dtype, generator=generator)
        return self.alphas[t].sqrt() * x + self.betas[t].sqrt() * z

def get_schedule_jump(T, jump_length=10, jump_n_sample=5):
    """RePaint'in resampling ('time-travel') şeması — resmî repo ile aynı.

    Dönen listede ardışık AZALAN çift = ters difüzyon adımı, ARTAN çift = ileri zıplama.
    Zıplamalar üretilen bölgenin bilinen bölgeyle ANLAMCA uyumlanmasını sağlar; makalenin
    ana katkısı bu (zıplamasız versiyon sadece dokuca uyar)."""
    jumps = {j: jump_n_sample - 1 for j in range(0, T - jump_length, jump_length)}
    t, ts = T, []
    while t >= 1:
        t -= 1; ts.append(t)
        if jumps.get(t, 0) > 0:
            jumps[t] -= 1
            for _ in range(jump_length):
                t += 1; ts.append(t)
    ts.append(-1)
    return ts

DIF = Diffusion(T=200)
print("T =", DIF.T, "| RePaint adım sayısı (j=10, U=3):", len(get_schedule_jump(DIF.T, 10, 3)))

In [ ]:
# ---------------- D2 · denoiser (PARÇASIZ) ----------------
# Part notebook'undaki ile tek farkı: parça one-hot girdisi ve part_pool YOK.
# Geriye kalan: EdgeConv (yerel geometri) + global havuz, t ile FiLM.
def knn_graph(xyz, k, chunk=4096):
    """(B,N,3) -> (B,N,k) komşu indisleri (kendisi hariç), sorgu üzerinden parçalı."""
    B, N, _ = xyz.shape
    out = torch.empty(B, N, k, dtype=torch.long, device=xyz.device)
    kk = min(k + 1, N)
    for s in range(0, N, chunk):
        d = torch.cdist(xyz[:, s:s + chunk], xyz)
        idx = d.topk(kk, dim=-1, largest=False).indices[:, :, 1:]
        if idx.shape[-1] < k:
            idx = idx[..., [i % idx.shape[-1] for i in range(k)]]
        out[:, s:s + chunk] = idx
    return out

def _gather_nb(h, idx):                        # h (B,N,C), idx (B,N,k) -> (B,N,k,C)
    B, N, C = h.shape; k = idx.shape[-1]
    off = (torch.arange(B, device=h.device) * N).view(B, 1, 1)
    return h.reshape(B * N, C)[(idx + off).reshape(-1)].reshape(B, N, k, C)

def timestep_embedding(t, dim):
    half = dim // 2
    f = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
    a = t.float().view(-1, 1) * f.view(1, -1)
    return torch.cat([a.sin(), a.cos()], -1)

class Block(nn.Module):
    """EdgeConv (yerel geometri) + global havuz, t ile FiLM'lenir."""
    def __init__(self, w):
        super().__init__()
        self.edge = nn.Sequential(nn.Linear(2 * w + 4, w), nn.GELU(), nn.Linear(w, w))
        self.fuse = nn.Sequential(nn.LayerNorm(2 * w), nn.Linear(2 * w, w), nn.GELU(),
                                  nn.Linear(w, w))
        self.film = nn.Linear(w, 2 * w)
    def forward(self, h, idx, rel, temb):
        hj = _gather_nb(h, idx); hi = h.unsqueeze(2).expand_as(hj)
        e = self.edge(torch.cat([hi, hj - hi, rel], -1)).max(2).values
        g = h.max(1, keepdim=True).values.expand_as(h)
        d = self.fuse(torch.cat([e, g], -1))
        sc, sh = self.film(temb).unsqueeze(1).chunk(2, -1)
        return h + d * (1 + sc) + sh

class ColorDenoiser(nn.Module):
    """Nokta başına renk alanı için eps-tahmini; SADECE xyz'ye koşullu (parça yok).

    Permütasyona eşdeğişken, N'den bağımsız: 2048 noktalı GT'de eğitilir, ~7k noktalı
    birleşim bulutunda çalışır. Part'lı sürümden tek farkı parça yolunun (one-hot + part_pool)
    olmaması; bu ~%10 daha az parametre demek. Fark kapasiteden değil KOŞULLAMADAN gelsin
    istiyorsan width'i 132-136 yapıp eşitleyebilirsin — varsayılan olarak eşitlemedim çünkü
    %10 kapasite bu ölçekte belirleyici değil ve mimariyi birebir karşılaştırılabilir tutmak
    daha okunaklı.
    """
    def __init__(self, width=128, k=16, n_blocks=3):
        super().__init__()
        self.k, self.width = k, width
        self.inp = nn.Linear(3 + 3, width)                      # xyz, c_t
        self.temb = nn.Sequential(nn.Linear(width, width), nn.SiLU(), nn.Linear(width, width))
        self.blocks = nn.ModuleList([Block(width) for _ in range(n_blocks)])
        self.out = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, width), nn.GELU(),
                                 nn.Linear(width, 3))
    def build_ctx(self, xyz):
        """Geometri her difüzyon adımında SABİT -> kNN grafiği bir kez kurulur."""
        idx = knn_graph(xyz, self.k)
        rel = _gather_nb(xyz, idx) - xyz.unsqueeze(2)
        scale = rel.norm(dim=-1).mean(dim=(1, 2), keepdim=True).clamp(min=1e-6).unsqueeze(-1)
        rel = torch.cat([rel / scale, rel.norm(dim=-1, keepdim=True) / scale], -1)
        return dict(xyz=xyz, idx=idx, rel=rel)
    def forward(self, c_t, t, ctx):
        B = c_t.shape[0]
        h = self.inp(torch.cat([ctx["xyz"], c_t], -1))
        temb = self.temb(timestep_embedding(t.expand(B) if t.dim() else t.repeat(B), self.width))
        for blk in self.blocks:
            h = blk(h, ctx["idx"], ctx["rel"], temb)
        return self.out(h)

print("parçasız denoiser tanımlı |",
      round(sum(p.numel() for p in ColorDenoiser().parameters()) / 1e6, 3), "M param")

In [ ]:
# ---------------- D3 · KOŞULSUZ eğitim (tek model) ----------------
def train_color_ddpm(model, dif, clouds, epochs=300, bs=8, lr=2e-4, device=DEV, log=25):
    """TAM renkli bulutlarda koşulsuz DDPM eğitimi — occlusion maskesi hiç görülmez.
       Part notebook'undaki ile aynı; sadece parça etiketi beslenmiyor."""
    model.to(device).train()
    opt = torch.optim.AdamW(model.parameters(), lr, weight_decay=1e-4)
    n, hist = len(clouds), []
    for ep in range(epochs):
        perm = np.random.permutation(n); tot = 0.0
        for s in range(0, n, bs):
            b = [clouds[i] for i in perm[s:s + bs]]
            xyz = torch.stack([torch.as_tensor(d["xyz"]) for d in b]).float().to(device)
            rgb = torch.stack([torch.as_tensor(d["rgb"]) for d in b]).float().to(device)
            x0 = rgb * 2 - 1
            t = torch.randint(0, dif.T, (len(b),), device=device)
            noise = torch.randn_like(x0)
            loss = ((model(dif.q_sample(x0, t, noise), t, model.build_ctx(xyz)) - noise) ** 2).mean()
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(b)
        hist.append(tot / n)
        if log and (ep % log == 0 or ep == epochs - 1):
            print(f"  ep{ep:4d}  eps-MSE {hist[-1]:.4f}")
    return hist

TRAIN_CLOUDS = [dict(xyz=DATA[i]["gt"][:, :3], rgb=DATA[i]["gt"][:, 3:6]) for i in TRAIN_IDX]
DDPM_EPOCHS = 300

ddpm = ColorDenoiser(width=128, k=16, n_blocks=3)
print("── RePaint (parçasız) eğitimi")
train_color_ddpm(ddpm, DIF, TRAIN_CLOUDS, epochs=DDPM_EPOCHS, bs=8, log=50)

In [ ]:
# ---------------- D4 · RePaint çıkarımı (parçasız) ----------------
@torch.no_grad()
def repaint_colors(model, dif, xyz, known, known_rgb, *, jump_length=10, jump_n_sample=3,
                   seed=0, device=DEV, return_diag=False):
    """Düz RePaint (Lugmayr et al. 2022): doldurulan noktaların rengini inpaint eder.

        x_{t-1} = m . q(x_bilinen, t-1)  +  (1-m) . p_theta(x_t)

    Part notebook'undaki sürümden farkları: parça bazlı dayanak muhasebesi yok,
    *anchorless* tespiti yok, uyarlanır resampling yok (jump_n_sample sabit).
    xyz (N,3), known (N,) bool, known_rgb (N,3) [0,1] -> (N,3) [0,1].
    """
    g = torch.Generator(device=device).manual_seed(seed)
    xyz_t = torch.as_tensor(xyz).float().unsqueeze(0).to(device)
    m = torch.as_tensor(known).bool().view(1, -1, 1).to(device)
    c0 = (torch.as_tensor(known_rgb).float().unsqueeze(0).to(device) * 2 - 1) * m

    model.eval().to(device)
    ctx = model.build_ctx(xyz_t)
    x = torch.randn(1, xyz_t.shape[1], 3, device=device, generator=g)

    ts = get_schedule_jump(dif.T, jump_length, jump_n_sample)
    for t_cur, t_next in zip(ts[:-1], ts[1:]):
        if t_next < t_cur:                                          # ters adım
            eps = model(x, torch.tensor(t_cur, device=device), ctx)
            x_unknown, _ = dif.p_sample(eps, x, t_cur, generator=g)
            if t_cur > 0:
                noise = torch.randn(x.shape, device=device, generator=g)
                x_known = dif.q_sample(c0, torch.tensor([t_cur - 1], device=device), noise)
            else:
                x_known = c0
            x = torch.where(m, x_known, x_unknown)
        else:                                                       # ileri zıplama
            x = dif.forward_jump(x, t_cur, generator=g)

    out = ((x[0] + 1) / 2).clamp(0, 1).cpu().numpy()
    kn = np.asarray(known, bool)
    out[kn] = np.asarray(known_rgb, np.float32)[kn]                 # görünür renkler korunur
    if return_diag:
        return out, dict(visible_ratio=float(kn.mean()), jump_n_sample=jump_n_sample,
                         n_steps=len(ts))
    return out


def make_repaint_input(d, comp, drop_part=None):
    """Birleşim bulutu = görünür partial (renk BİLİNEN) + PoinTr'ın doldurduğu (BİLİNMEYEN).

    drop_part: o parçanın görünür noktaları da maskelenir -> 'tamamen kapalı bölge' senaryosu.
    Düşürme GT etiketine göre (deneyi tanımlamak için); yöntem bu etiketi GÖRMÜYOR.
    """
    partial = d["partial"]
    xyz = np.concatenate([partial[:, :3], comp], 0).astype(np.float32)
    c = xyz.mean(0)
    xyz = ((xyz - c) / (np.linalg.norm(xyz - c, axis=1).max() + 1e-9)).astype(np.float32)
    known = np.zeros(len(xyz), bool); known[:len(partial)] = True
    rgb = np.zeros((len(xyz), 3), np.float32); rgb[:len(partial)] = partial[:, 3:6]
    if drop_part is not None:
        known &= (segment_oracle(d, xyz) != drop_part); rgb[~known] = 0
    return dict(xyz=xyz, rgb=rgb, known=known, n_vis=len(partial))

print("parçasız RePaint çıkarımı hazır")

---
# E · Değerlendirme

Ölçüt, seed'ler ve test bölümü part notebook'uyla **aynı** — satırlar doğrudan yan yana konabilir.

In [ ]:
# ---------------- E1 · ana tablo (test bölümü) ----------------
EVAL_N, N_SEEDS = 10, 2

_keys = ["NN-kopya"] + (["parça-ortalaması"] if PART_BASELINES else []) + ["RePaint (parçasız)",
                                                                          "oracle tavan"]
rows = {k: [] for k in _keys}; sd_rp = []
for n, i in enumerate(TEST_IDX[:EVAL_N]):
    d = DATA[i]; gt, partial = d["gt"], d["partial"]
    comp = complete_of(d)
    _, gi = cKDTree(gt[:, :3]).query(comp[:, :3], k=1)
    true_rgb = gt[gi, 3:6]
    m = d["miss"][gi]; m = m if m.any() else np.ones(len(comp), bool)

    rows["NN-kopya"].append(deltaE(nn_color(partial, comp)[m], true_rgb[m]).mean())
    if PART_BASELINES:
        rows["parça-ortalaması"].append(
            deltaE(part_color_pointnet(partial, comp, seg_model)[m], true_rgb[m]).mean())
    pf = np.stack([gt[d["gt_part"] == p, 3:6].mean(0) if (d["gt_part"] == p).any()
                   else gt[:, 3:6].mean(0) for p in range(NUM_PARTS)])
    rows["oracle tavan"].append(deltaE(pf[d["gt_part"][gi]][m], true_rgb[m]).mean())

    inp = make_repaint_input(d, comp)
    s = [deltaE(repaint_colors(ddpm, DIF, inp["xyz"], inp["known"], inp["rgb"],
                               seed=1000 * n + k)[inp["n_vis"]:][m], true_rgb[m]).mean()
         for k in range(N_SEEDS)]
    rows["RePaint (parçasız)"].append(float(np.mean(s))); sd_rp.append(float(np.std(s)))
    print(f"  [{n+1}/{min(EVAL_N,len(TEST_IDX))}] model {i} bitti", flush=True)

print(f"\neksik-bölge ΔE(Lab) — {min(EVAL_N,len(TEST_IDX))} GÖRÜLMEMİŞ model, occlusion {CROP:.0%}")
print("-" * 58)
for k, v in rows.items():
    v = np.array(v)
    extra = f"  (örnekler-arası std {np.mean(sd_rp):.2f})" if k.startswith("RePaint") else ""
    print(f"  {k:<20} {v.mean():7.3f}  ± {v.std():5.3f}{extra}")
print("-" * 58)
a, b = np.array(rows["NN-kopya"]), np.array(rows["RePaint (parçasız)"])
print(f"  RePaint, NN-kopyayı {(b < a).sum()}/{len(a)} modelde yendi "
      f"({a.mean()/max(b.mean(),1e-9):.2f}x ortalama)")
if PART_BASELINES:
    c = np.array(rows["parça-ortalaması"])
    print(f"  parça-ortalamasına karşı: {(b < c).sum()}/{len(c)} modelde daha iyi")
    print("\n  >>> Bu satırı part notebook'undaki 'RePaint-part' ile karşılaştır:")
    print("      fark = parça koşullamasının ÖLÇÜLMÜŞ katkısı.")

In [ ]:
# ---------------- E2 · TAMAMEN KAPALI bölge ----------------
# Part notebook'uyla AYNI senaryo, aynı hedef parça -> satırlar doğrudan kıyaslanabilir.
# GT etiketi sadece hangi bölgenin kapatılacağını tanımlıyor; yöntem onu görmüyor.
cnt_all = np.bincount(np.concatenate([DATA[i]["gt_part"] for i in TEST_IDX[:EVAL_N]]),
                      minlength=NUM_PARTS)
TGT = int(np.argsort(-cnt_all)[1]) if NUM_PARTS > 1 else 0

_k2 = ["NN-kopya"] + (["parça-ortalaması"] if PART_BASELINES else []) + ["RePaint (parçasız)"]
res = {k: [] for k in _k2}; n_case = 0
for i in TEST_IDX[:EVAL_N]:
    d = DATA[i]; gt, partial = d["gt"], d["partial"]
    comp = complete_of(d)
    _, gi = cKDTree(gt[:, :3]).query(comp[:, :3], k=1)
    true_rgb, gp = gt[gi, 3:6], d["gt_part"][gi]
    sel = gp == TGT
    if sel.sum() < 20: continue
    n_case += 1

    inp = make_repaint_input(d, comp, drop_part=TGT)
    kn = inp["known"][:inp["n_vis"]]
    vis_xyz, vis_rgb = inp["xyz"][:inp["n_vis"]][kn], inp["rgb"][:inp["n_vis"]][kn]
    _, j = cKDTree(vis_xyz).query(inp["xyz"][inp["n_vis"]:], k=1)
    res["NN-kopya"].append(deltaE(vis_rgb[j][sel], true_rgb[sel]).mean())
    if PART_BASELINES:
        _seg = segment(seg_model, inp["xyz"])
        vl, cl = _seg[:inp["n_vis"]][kn], _seg[inp["n_vis"]:]
        pm = np.tile(vis_rgb.mean(0), (NUM_PARTS, 1))
        for k in range(NUM_PARTS):
            if (vl == k).any(): pm[k] = vis_rgb[vl == k].mean(0)
        res["parça-ortalaması"].append(deltaE(pm[cl][sel], true_rgb[sel]).mean())
    rp, diag = repaint_colors(ddpm, DIF, inp["xyz"], inp["known"], inp["rgb"],
                              seed=i, return_diag=True)
    res["RePaint (parçasız)"].append(deltaE(rp[inp["n_vis"]:][sel], true_rgb[sel]).mean())
    if n_case == 1: print("diag:", diag)

print(f"\nTAMAMEN KAPALI '{PART_NAMES[TGT]}' bölgesinde ΔE(Lab) — {n_case} model")
print("-" * 58)
for k, v in res.items(): print(f"  {k:<20} {np.mean(v):7.3f}  ± {np.std(v):5.3f}")
print("-" * 58)
print("  Parçasız model o bölgenin NE OLDUĞUNU bilmiyor; sadece geometri + komşu renkler.")
print("  Part notebook'undaki karşılığıyla kıyasla — asıl fark burada çıkmalı.")

In [ ]:
# ---------------- E3 · görsel karşılaştırma ----------------
import plotly.graph_objects as go
from plotly.subplots import make_subplots

i = TEST_IDX[0]; d = DATA[i]; gt, partial = d["gt"], d["partial"]
comp = complete_of(d)
inp = make_repaint_input(d, comp)
rp = repaint_colors(ddpm, DIF, inp["xyz"], inp["known"], inp["rgb"], seed=0)
nn_rgb = nn_color(partial, comp)

def tr(xyz, rgb, size=1.6):
    c = ["rgb(%d,%d,%d)" % tuple((np.clip(x, 0, 1) * 255).astype(int)) for x in rgb]
    return go.Scatter3d(x=xyz[:,0], y=xyz[:,1], z=xyz[:,2], mode="markers",
                        marker=dict(size=size, color=c))
def full(rgb_comp):
    return np.vstack([partial[:, :3], comp]), np.vstack([partial[:, 3:6], rgb_comp])

panels = [("occluded (girdi)", (partial[:, :3], partial[:, 3:6])), ("NN-kopya", full(nn_rgb))]
if PART_BASELINES:
    panels.append(("parça-ortalaması", full(part_color_pointnet(partial, comp, seg_model))))
panels += [("RePaint (parçasız)", full(rp[inp["n_vis"]:])), ("GT", (gt[:, :3], gt[:, 3:6]))]

fig = make_subplots(rows=1, cols=len(panels), specs=[[{"type": "scene"}] * len(panels)],
                    subplot_titles=[t for t, _ in panels])
for c, (_, (x, r)) in enumerate(panels, 1): fig.add_trace(tr(x, r), 1, c)
for s in fig.layout:
    if s.startswith("scene"): fig.layout[s].aspectmode = "data"
fig.update_layout(height=430, showlegend=False, margin=dict(l=0, r=0, t=30, b=0)); fig.show()

---
## İki notebook'u nasıl okumalı

Aynı veri, aynı seed'ler, aynı metrik olduğu için satırlar doğrudan yan yana konabilir:

| satır | nereden |
|---|---|
| NN-kopya | ikisinde de aynı çıkmalı (sağlama) |
| parça-ortalaması | ikisinde de aynı çıkmalı (sağlama) |
| oracle tavan | ikisinde de aynı çıkmalı (sağlama) |
| **RePaint (parçasız)** | bu notebook |
| **RePaint-part** | part notebook'u |
| **RePaint-part (oracle seg)** | part notebook'u |

İlk üç satırın iki notebook'ta **aynı** çıkması senin sağlamandır — tutmuyorsa
veri/seed ayarlarından biri kaymış demektir.

Sonra okunacak iki fark:

* `RePaint-part` − `RePaint (parçasız)` = **parça koşullamasının gerçek koşullardaki katkısı**
  (gürültülü PointNet etiketleriyle).
* `RePaint-part (oracle seg)` − `RePaint (parçasız)` = **fikrin tavanı**, mükemmel etiketle.

Aradaki boşluk part-seg'i iyileştirmenin ne kazandıracağını söyler. Yerel testte bu boşluk
büyüktü (32.7 vs 20.1) — yani en kârlı bir sonraki iş difüzyonu değil, **segmenter'ı**
iyileştirmek olabilir.